# Monotonic Nonlinear State-Space (MNSS) Model Architecture

While the `career_recommender_model.ipynb` handles the cross-sectional O-Level clustering (initializing $z_1$), this notebook implements the advanced, graduate-level deep learning architecture for the **longitudinal tracking** and **skill gap optimization** described in the thesis reference paper.

Here, we implement the core mathematical innovations in PyTorch:
1. **Monotonic GRU Cell:** Ensuring hidden skills never decrease.
2. **The ELBO Objective:** Variational inference for learning latent skill states.
3. **Skill Gap Optimization:** Using projected gradient descent with an $L_1$ penalty.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

## 1. The Monotonic GRU Cell

Standard RNNs allow hidden states to fluctuate up and down. The MNSS paper introduces a **Monotonic GRU** to enforce the assumption that skills only increase (a user never forgets a skill over time).

The core update equation is:
$\gamma_t = \gamma_{t-1} + (1 - \gamma_{t-1}) \odot o_t$

Where $(1 - \gamma_{t-1})$ represents the remaining skill gap, and $o_t$ is the portion of that gap filled by the new job experience.

In [4]:
class MonotonicGRUCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(MonotonicGRUCell, self).__init__()
        self.hidden_size = hidden_size
        
        # Update gate parameters (k_t)
        self.W_k = nn.Linear(input_size, hidden_size, bias=False)
        self.U_k = nn.Linear(hidden_size, hidden_size, bias=False)
        
        # Reset gate parameters (r_t)
        self.W_r = nn.Linear(input_size, hidden_size, bias=False)
        self.U_r = nn.Linear(hidden_size, hidden_size, bias=False)
        
        # Update value parameters (o_t)
        self.W_o = nn.Linear(input_size, hidden_size, bias=False)
        self.U_o = nn.Linear(hidden_size, hidden_size, bias=False)
        
    def forward(self, u_t, gamma_prev):
        """
        u_t: Current career experience embedding (batch_size, input_size)
        gamma_prev: Previous latent skill state (batch_size, hidden_size) where values are in [0, 1]
        """
        # Gates
        k_t = torch.sigmoid(self.W_k(u_t) + self.U_k(gamma_prev))
        r_t = torch.sigmoid(self.W_r(u_t) + self.U_r(gamma_prev))
        
        # The portion of the skill deficiency gap being filled
        o_t = torch.sigmoid(self.W_o(u_t) + self.U_o(r_t * gamma_prev)) * k_t
        
        # Monotonic Update Rule: mastery increases, bounded by 1.0
        # This directly implements diminishing returns.
        gamma_t = gamma_prev + (1.0 - gamma_prev) * o_t
        
        return gamma_t

## 2. The ELBO Objective (Variational Inference)

We cannot observe the hidden skills directly. The paper uses Variational Inference, maximizing the Evidence Lower Bound (ELBO) to force the hidden states to accurately reconstruct the observed jobs.

$ELBO = \mathbb{E}[\log P(x|z)] - D_{KL}(q || P)$

In [5]:
def calculate_elbo_loss(reconstruction_logits, target_jobs, q_gamma, p_gamma):
    """
    reconstruction_logits: The predicted job probabilities from the Emission model
    target_jobs: The actual observed jobs
    q_gamma: The approximate posterior skills (from Recognition network)
    p_gamma: The prior skills (from Prior network)
    """
    # 1. Reconstruction Loss: E[log P(x|z)]
    # Typically implemented as Cross Entropy (negative log likelihood)
    recon_loss = F.cross_entropy(reconstruction_logits, target_jobs)
    
    # 2. KL Divergence between the two Bernoulli distributions (q and p)
    # KL(q || p) = q * log(q/p) + (1-q) * log((1-q)/(1-p))
    epsilon = 1e-7 # for numerical stability
    q = torch.clamp(q_gamma, epsilon, 1.0 - epsilon)
    p = torch.clamp(p_gamma, epsilon, 1.0 - epsilon)
    
    kl_divergence = torch.sum(q * torch.log(q / p) + (1 - q) * torch.log((1 - q) / (1 - p)), dim=1).mean()
    
    # Total ELBO Loss (we want to minimize negative ELBO)
    # Note: beta is a hyperparameter from the paper balancing KL and Reconstruction
    beta = 0.1
    loss = recon_loss + beta * kl_divergence
    
    return loss, recon_loss, kl_divergence

## 3. Skill Gap Optimization

This is the algorithm to provide actionable feedback to a student. Given their current skills ($z_t$) and a target career goal, we find the mathematically smallest set of skills they need to learn to get the job.

Objective: $\min_z [ -\log P(Goal|z) + \lambda ||z - z_t||_1 ]$

In [7]:
def optimize_skill_gap(current_skills, target_job_id, emission_model, lambda_reg=0.1, lr=0.01, iterations=100):
    """
    Finds the optimal new skill state (z_star) to reach a target job.
    """
    # We want to optimize a new skill vector, initialized to current skills
    # z_star must be greater than or equal to current_skills (Monotonicity)
    z_star = current_skills.clone().detach().requires_grad_(True)
    
    optimizer = optim.Adam([z_star], lr=lr)
    
    for i in range(iterations):
        optimizer.zero_grad()
        
        # Forward pass through the emission model to get job probabilities
        job_logits = emission_model(z_star)
        
        # 1. Maximize probability of reaching the target job (Minimize NLL)
        target_tensor = torch.tensor([target_job_id])
        nll_loss = F.cross_entropy(job_logits.unsqueeze(0), target_tensor)
        
        # 2. L1 Penalty: Keep the skill changes sparse (don't recommend learning everything)
        # Only penalize the difference added
        l1_penalty = lambda_reg * torch.sum(torch.abs(z_star - current_skills))
        
        loss = nll_loss + l1_penalty
        loss.backward()
        optimizer.step()
        
        # Projected Gradient Descent: Ensure z_star stays in [0, 1] AND is >= current_skills
        with torch.no_grad():
            # Clamp to current_skills as the minimum, and 1.0 as maximum
            z_star.data = torch.max(z_star.data, current_skills)
            z_star.data = torch.min(z_star.data, torch.ones_like(z_star))
            
    return z_star.detach()

print("MNSS PyTorch Architecture Defined. Ready for longitudinal training.")

MNSS PyTorch Architecture Defined. Ready for longitudinal training.
